# Fine-tuning QLoRA — Assistente Medico Virtual (Tech Challenge Fase 3)

Notebook pronto para rodar o fine-tuning do `Qwen/Qwen2.5-3B-Instruct` com o dataset sintetico do hospital ficticio (`data/raw/faqs`, `data/raw/laudos_modelo`), usando QLoRA em 4-bit. Precisa de GPU — no Colab gratuito, ative T4 antes de continuar:

**Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU (T4)**

Este notebook trabalha DIRETO na copia do projeto salva no seu Google Drive, para que o dataset processado, o indice do RAG e o adapter treinado fiquem salvos ali — se a sessao do Colab cair ou expirar, nada se perde (ver PLANO_Fase3.md, secao 8, Riscos).

## 1. Verificar GPU

Se a saida abaixo nao mostrar uma GPU (ex.: Tesla T4), va em **Ambiente de execucao > Alterar tipo de ambiente de execucao** e selecione uma GPU antes de continuar — sem isso o fine-tuning nao roda (ver aviso no topo de `finetuning/train_qlora.py`).

In [ ]:
!nvidia-smi

## 2. Montar o Google Drive

Vai pedir autorizacao na primeira vez.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Localizar (ou copiar) o projeto dentro do Drive

Ajuste `PROJECT_DIR` abaixo para o caminho, dentro do seu Drive, onde a pasta `1.AssistenteMedico` esta (ou vai ficar). Se ainda nao subiu a pasta para o Drive, duas opcoes:

- **Mais simples:** faca upload manual da pasta `1.AssistenteMedico` (a do seu repositorio local) para o caminho abaixo pelo painel de arquivos do Google Drive (drive.google.com), antes de rodar a celula seguinte.
- **Via git:** se o repositorio `AI-TechChallenge-Step3` estiver no GitHub, descomente e ajuste a linha `!git clone ...` na celula abaixo em vez de fazer upload manual.

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/TechChallengeFase3/1.AssistenteMedico'

# Alternativa via git (descomente e ajuste a URL se o repo estiver no GitHub):
# !git clone <URL_DO_SEU_REPOSITORIO> /content/drive/MyDrive/TechChallengeFase3

import os
assert os.path.isdir(PROJECT_DIR), (
    f'Pasta nao encontrada: {PROJECT_DIR}. Faca upload da pasta 1.AssistenteMedico '
    'para esse caminho no seu Google Drive (ou ajuste PROJECT_DIR acima) antes de continuar.'
)
%cd {PROJECT_DIR}
!ls

## 4. Instalar dependencias

O Colab ja vem com `torch` e CUDA configurados — instalamos so o resto do `requirements.txt`. Pode levar 2-3 minutos na primeira vez.

In [ ]:
!pip install -q -r requirements.txt

## 5. Preparar o dataset (se ainda nao tiver rodado)

Gera `data/processed/dataset_train.jsonl` e `dataset_val.jsonl` a partir dos .md em `data/raw/faqs/` e `data/raw/laudos_modelo/`. Seguro rodar de novo — regenera os arquivos do zero.

In [ ]:
!python finetuning/prepare_dataset.py

## 6. Rodar o fine-tuning QLoRA

Com GPU ativa, isso deve funcionar sem cair no guard de `_check_gpu_available()`. Duracao esperada: poucos minutos, dado o tamanho do dataset sintetico atual (17 exemplos) — cresce conforme o dataset crescer.

O adapter e salvo em `finetuning/adapters/cancer-assistant-lora/` — como estamos rodando dentro do Drive montado, isso ja fica persistido automaticamente.

In [ ]:
!python finetuning/train_qlora.py

## 7. Teste rapido do adapter treinado

Carrega o modelo base + adapter e gera uma resposta para uma pergunta de teste, so para conferir que o fine-tuning teve algum efeito perceptivel — a comparacao formal base vs. fine-tuned fica em `finetuning/evaluate.py`.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from finetuning.config import ADAPTER_OUTPUT_DIR, BASE_MODEL_NAME, SYSTEM_PROMPT

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_OUTPUT_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, device_map='auto', torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_OUTPUT_DIR)

pergunta = 'Uma paciente teve resultado de mamografia com classificacao BI-RADS 4. Qual a conduta recomendada?'
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': pergunta},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(model.device)
saida = model.generate(inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
print(tokenizer.decode(saida[0][inputs.shape[-1]:], skip_special_tokens=True))

## Proximos passos

- Se o resultado parecer fraco, o dataset sintetico atual (17 exemplos) e pequeno de proposito para validar o pipeline — expandir `data/raw/faqs/` e `data/raw/laudos_modelo/` com mais exemplos deve ajudar (ver PLANO_Fase3.md, secao 3).
- Depois do fine-tuning, volte para `rag/chain.py` (`load_llm`) para plugar este adapter na chain de RAG.
- Rode `finetuning/evaluate.py` para a comparacao formal base vs. fine-tuned que vai para o relatorio tecnico.